# Phase 2 — Architecture des données

**Autrice :** Faty Barbosa Monteiro

**Date :** 22 octobre 2026

**Client :** CLIENT001

**Lu après :** `documentation/01_cadrage_projet.ipynb`

Dans un mandat d'investissement, la qualité du calcul dépend de l'organisation des données qui le précèdent. Cette phase fixe cette organisation avant l'acquisition et avant l'analyse.

Elle répond aux trois questions de l'architecture : d'où viennent les données, comment elles circulent, et où elles sont conservées.

La note tient les trois livrables de la phase : le schéma d'architecture, le flux de données et le dictionnaire de données.


## 1. Données requises et sources

Le mandat a besoin d'un nombre limité de familles de données. Chaque famille a une finalité et une source de référence. Une source ne tient pas lieu d'une autre.

| Famille | Donnée | Finalité | Source de référence |
|---|---|---|---|
| Identité | Identifiant FIGI | Reconnaître le même titre d'un fichier à l'autre | OpenFIGI |
| Identité | Ligne de cotation | Désigner le titre sur lequel porte le plafond de 5 % | Symbole de cotation, formé à partir du code et de la place |
| Identité | Dénomination | Nommer la société | Feuille des positions iShares |
| Identité | Place de cotation | Rattacher le titre à son marché | Feuille des positions iShares |
| Classification | Liste des titres | Constituer la liste examinée | Feuille des positions du fonds iShares MSCI Global Sustainable Development Goals ETF |
| Classification | Catégorie d'instrument | Retenir les actions cotées | Cette feuille. Elle ne contient que des actions |
| Classification | Secteur | Tenir le plafond de 20 % par métier | Cette feuille, dans la langue de la feuille. Un secteur absent est rangé dans Non classé |
| Classification | Pays | Tenir le plafond de 50 % par grande région | Cette feuille. Les régions sont Amériques, Europe, Asie-Pacifique, Moyen-Orient et Afrique. Un pays absent est rangé dans Non classé |
| Classification | Devise de cotation | Tenir le plafond de 40 % par monnaie et choisir la série de change | Monnaie du cours publié par Yahoo Finance |
| Marché | Cours de clôture | Mesurer la variation du prix | Yahoo Finance. Le cours retenu est le dernier prix de la séance, observé en fin de mois |
| Marché | Volume | Apprécier la liquidité d'une ligne | Yahoo Finance. Le volume est le nombre de titres échangés au cours de chaque séance. Une position ne dépasse pas 10 % du volume moyen sur 20 séances |
| Change | Cours contre le franc | Exprimer les prix et les résultats en francs | Banque nationale suisse |
| Taux | Taux sans risque | Mesurer le résultat au-dessus du taux | SARON, taux au jour le jour publié par la Banque nationale suisse |
| Comparaison | Rendement du fonds | Mesurer la distance entre le portefeuille et le fonds de comparaison | Rendement publié par iShares, converti en francs au cours de la Banque nationale suisse |
| Éligibilité | Activité économique | Constituer l'unité de preuve | Rapports annuels, rapports financiers, rapports de durabilité, documents réglementaires et publications officielles de la société |
| Éligibilité | Contribution à un ODD | Fonder l'entrée | Ces mêmes documents. La preuve retient le passage, la page et la date |
| Éligibilité | Atteinte à un ODD | Contrôler le non-préjudice | Ces mêmes documents. Une contribution positive ne compense pas une atteinte |
| Éligibilité | Controverse | Juger la nature, la gravité et la réponse de la société | D'abord les documents de la société, puis les sources publiques et institutionnelles |
| Éligibilité | Statut | Décider l'entrée | Registre constitué à partir des preuves. Les statuts sont Eligible, Not_Eligible, Indeterminate et Excluded. Indeterminate n'est pas une entrée |
| Exclusion | Activités écartées | Écarter le tabac, les armes conventionnelles, le charbon thermique et les jeux d'argent | Une source nommée, société par société. La méthodologie ne fixe pas de seuil de chiffre d'affaires |
| Émetteur | Informations fondamentales publiées | Documenter l'émetteur | Rapports financiers de la société et dépôts officiels accessibles via EDGAR |
| Macroéconomie | Séries macroéconomiques | Situer le contexte économique | FRED |

Le dividende versé par la société n'est pas une donnée à acquérir. Le rendement du portefeuille est la variation du cours de clôture converti en francs. Le rendement du fonds de comparaison est, lui, le rendement que le fonds publie.


### 1.1 Sources qui ne décident pas à la place d'une autre

La référence stratégique du mandat est le MSCI ACWI Sustainable Development Index (Net). Net signifie : dividendes après impôt. Cet indice désigne la liste à laquelle le mandat se compare. Il n'est pas acquis. Il ne constitue pas la source du cours, ni celle du rendement publié. La feuille de positions et le rendement publié sont ceux du fonds iShares qui suit cet indice. La présence dans ce fonds n'est pas une preuve d'éligibilité. L'absence n'est pas une preuve d'inéligibilité.

Les données ESG publiques, c'est-à-dire les notes environnementales, sociales et de gouvernance diffusées par des fournisseurs, ne constituent pas la décision d'entrée. Cette décision se lit dans les rapports de la société, au regard des Objectifs de développement durable (ODD).

Yahoo Finance publie également un cours ajusté, après dividendes et divisions de titres. Le rendement du mandat retient le cours de clôture.

La Banque mondiale et le FMI sont les autres sources publiques identifiées pour la famille macroéconomique. Elles ne fournissent ni le cours de change du mandat, ni le SARON. Le repli documentaire de FRED vers le FMI ne déclenche pas une lecture de substitution.

Le registre des sources inscrit le fournisseur des dépôts officiels sous le libellé SED EDGAR. Le connecteur correspondant est celui d'EDGAR.


## 2. Schéma d'architecture

### 2.1 Sources, traitements, restitution

```text
SOURCES                              TRAITEMENTS                 RESTITUTION
Yahoo Finance                        Acquisition                 Base SQLite
Banque nationale suisse              Nettoyage                   Tableaux de bord
iShares                              Contrôles qualité           Rapports
OpenFIGI                             Enrichissement
Rapports de société
Sources publiques et institutionnelles
Source d'exclusion, par société
EDGAR
FRED
Banque mondiale
FMI
```

Chaque donnée de la section 1 a sa source dans ce bloc. Yahoo Finance fournit le cours de clôture, le volume et la devise de cotation. La Banque nationale suisse fournit le change et le SARON. iShares fournit la liste, la dénomination, la place, le secteur, le pays, la catégorie d'instrument et le rendement publié du fonds. OpenFIGI fournit l'identifiant. Les rapports de société fournissent l'activité, la contribution, l'atteinte et la preuve. Les sources publiques et institutionnelles complètent la controverse lorsque le document de la société ne suffit pas. La source d'exclusion est nommée société par société pour le tabac, les armes conventionnelles, le charbon thermique et les jeux d'argent. EDGAR fournit les dépôts officiels. FRED fournit les séries macroéconomiques. La Banque mondiale et le FMI sont les autres sources publiques de cette même famille.

La ligne de cotation et le statut ODD ne sont pas des sources. La ligne est formée à partir du code et de la place. Le statut est établi à partir des preuves.

L'acquisition dépose le fichier tel que la source l'a publié. Le nettoyage le standardise. Les contrôles de qualité en vérifient la complétude, la cohérence, la validité et l'unicité. L'enrichissement ajoute une information déduite d'une source déjà détenue, notamment le cours exprimé en francs. Il ne crée ni un cours manquant, ni un statut manquant.

La restitution conserve les séries dans la base, les présente dans les tableaux de bord et les remet dans les rapports.


### 2.2 Zones de stockage

```text
documentation/          notes du mandat et des phases
data/
    01_raw/             fichiers reçus, sans transformation
        benchmark/      positions du fonds de comparaison
        securities/     identité des titres
        market/         cours et volume
        fundamentals/   informations fondamentales
        macro/          séries macroéconomiques et SARON
        fx/             cours de change
        odd/            registre d'éligibilité
        exclusions/     registre des exclusions
    02_processed/       fichiers nettoyés et enrichis
    03_universe/        liste examinée, titres retenus, titres écartés
    04_portfolios/      poids, rendements et comparaison
    05_monitoring/      suivi du portefeuille
    06_performance/     mesures de performance
database/               base SQLite du projet
dashboards/powerbi/     tableaux de bord Power BI
reports/                rapports et pages de restitution
audit/                  registre des sources et journal d'acquisition
src/                    traitements
tests/                  contrôles de reproductibilité
```


### 2.3 Rôle des couches

**data/01_raw/.** Zone des fichiers reçus. Aucune transformation n'y est appliquée. Le fichier d'origine reste disponible pour un contrôle.

**data/02_processed/.** Zone des fichiers nettoyés, validés et exprimés dans la forme utile au calcul. Le référentiel des titres et le cours en francs y sont constitués.

**data/03_universe/.** Zone de la liste dans laquelle les portefeuilles sont construits. Elle distingue les titres examinés, les titres retenus et les titres écartés. Le registre ODD et le registre des exclusions sont les sources de la décision d'entrée et de sortie. Ils ne se confondent pas avec le seul contrôle de conversion en francs.

**data/04_portfolios/, data/05_monitoring/ et data/06_performance/.** Zones des résultats : poids, situation du portefeuille et mesures. Ce ne sont pas des fichiers de source.

**database/.** Base SQLite dans laquelle les jeux dont le chargement est prévu sont conservés pour les requêtes. Le fichier est `database/sdg_investment.db`.

**dashboards/powerbi/ et reports/.** Canaux de restitution. L'architecture réserve Power BI aux tableaux de bord. Les rapports rassemblent les commentaires et les pages remises au lecteur. Les pages produites à ce jour sont les HTML de `reports/`.

**audit/.** Gouvernance des sources et trace des acquisitions. Le registre des sources fait autorité sur l'origine de chaque famille.


## 3. Flux de données

Chaque étape suit le même enchaînement : la donnée entre, un traitement la prend, un résultat en sort, ce résultat est rangé, et l'étape suivante le consomme.

```text
ÉTAPE 1 — ACQUISITION
Sources de référence
        ↓
Acquisition
        ↓
Fichiers reçus
data/01_raw/
        ↓
Journal d'acquisition
audit/


ÉTAPE 2 — CONTRÔLES QUALITÉ
Fichiers reçus
        ↓
Complétude, cohérence, validité, unicité
        ↓
Fichiers acceptés pour la préparation


ÉTAPE 3 — NETTOYAGE ET ENRICHISSEMENT
Fichiers acceptés
Identité OpenFIGI
Cours de change de la Banque nationale suisse
        ↓
Référentiel des titres
Cours de clôture exprimé en francs
Volume conservé
        ↓
data/02_processed/


ÉTAPE 4 — LISTE INVESTISSABLE
Référentiel
Liste de départ iShares
Registre ODD
Registre des exclusions
        ↓
Titres examinés, retenus et écartés
        ↓
data/03_universe/


ÉTAPE 5 — RESTITUTION
Liste investissable
Cours en francs
SARON
Rendement publié du fonds, en francs
        ↓
Portefeuilles, suivi, performance
        ↓
Base SQLite
Tableaux de bord
Rapports
```

L'analyse du risque et les tests de résistance s'appuient sur les mêmes séries. Ils ne créent pas une source nouvelle.


## 4. Dictionnaire de données

Le dictionnaire décrit les champs que l'architecture retient pour chaque famille. Pour chaque champ, il précise le sens et l'usage. Il ne crée pas une colonne qu'aucune source ne publie.

### 4.1 Registre des sources

Le registre est `audit/source_registry.xlsx`. Il nomme la source et le jeu de données.

| Champ | Description | Utilisation |
|---|---|---|
| Source ID | Identifiant de la source. | Rattacher chaque fichier à son origine. |
| Provider | Nom du fournisseur, tel qu'il est inscrit. | Citer la source sans la rebaptiser. |
| URL | Adresse de la source. | Reprendre la donnée à l'endroit publié. |
| Reliability | Niveau de confiance inscrit. | Distinguer une source officielle d'une source de contexte. |
| Backup Source | Repli documentaire. | Signaler une source alternative. Ce champ ne lance pas une lecture. |
| Status | Situation de la ligne. | Savoir si la source est en service dans le registre. |
| Dataset ID | Identifiant du jeu. | Désigner un fichier précis. |
| Production Method | Mode de production : téléchargement, API, paquet ou préparation. | Décrire comment le fichier est obtenu. |
| Connector | Traitement qui lit la source, lorsqu'il existe. | Relier la source au programme d'acquisition. |
| Output File Name | Nom du fichier. | Retrouver le fichier produit. |
| Storage Directory | Dossier de conservation. | Ranger le fichier dans la bonne couche. |
| Frequency | Rythme de la série. | Distinguer le quotidien, le mensuel et l'annuel. |


### 4.2 Référentiel des titres

La feuille de positions iShares fournit le nom, la place, le secteur, le pays et la catégorie. OpenFIGI fournit l'identifiant stable. Yahoo Finance fournit le symbole de cotation et la monnaie du cours.

| Champ | Description | Utilisation |
|---|---|---|
| Ticker | Code du titre sur une place. | Reconnaître le titre dans la liste de départ. |
| Name | Dénomination de la société. | Lire les rapports et les portefeuilles. |
| Exchange | Place de cotation. | Rattacher le titre à son marché. |
| Secteur | Métier indiqué sur la feuille de positions, dans la langue de cette feuille. | Tenir le plafond de 20 %. Un secteur absent est rangé dans Non classé. |
| Pays | Pays inscrit sur la feuille de positions. | Rattacher le titre à une zone : Amériques, Europe, Asie-Pacifique, Moyen-Orient et Afrique. Un pays absent est rangé dans Non classé. |
| Devise de cotation | Monnaie du cours publié. | Tenir le plafond de 40 % et choisir la série de change. |
| FIGI | Financial Instrument Global Identifier, identifiant stable publié par OpenFIGI. | Relier les fichiers entre eux. |
| Symbole de cotation | Identifiant de la ligne de cours. | Demander le cours et le volume à Yahoo Finance. |
| Asset Class | Catégorie de l'instrument. | Vérifier que la ligne appartient à l'univers d'actions du mandat. |

L'ISIN, identifiant international d'un instrument, est retenu lorsqu'un document le publie. Il n'est pas la clé du référentiel lorsque la source ne le fournit pas.


### 4.3 Cours et volume

Source de référence : Yahoo Finance.

| Champ | Description | Utilisation |
|---|---|---|
| Date | Jour de la séance. | Construire la série. |
| Open | Premier prix de la séance. | Contrôler la cohérence de la séance. |
| High | Prix le plus haut de la séance. | Contrôler la cohérence de la séance. |
| Low | Prix le plus bas de la séance. | Contrôler la cohérence de la séance. |
| Close | Dernier prix de la séance. | Calculer le rendement à partir du cours de clôture de fin de mois, converti en francs. |
| Volume | Nombre de titres échangés pendant la séance. | Apprécier la liquidité. Le volume n'est pas un prix. |

Le fournisseur publie aussi un cours ajusté. Ce cours n'est pas le cours de clôture retenu pour le rendement du mandat.


### 4.4 Change, taux sans risque et macroéconomie

| Champ | Description | Utilisation |
|---|---|---|
| Date | Date du cours ou du taux. | Aligner le change, le taux et le cours du titre. |
| Devise | Monnaie cotée contre le franc. | Choisir la série de change. |
| Cours contre le franc | Prix de la monnaie publié par la Banque nationale suisse. | Convertir un prix étranger en francs. |
| SARON | Taux au jour le jour publié par la Banque nationale suisse. | Mesurer le résultat au-dessus du taux sans risque. |
| Série macroéconomique | Identifiant, date et valeur d'un indicateur de FRED. | Situer le contexte économique. |

La série macroéconomique ne remplace ni le change, ni le SARON.


### 4.5 Éligibilité, exclusions et fondamentaux

| Champ | Description | Utilisation |
|---|---|---|
| Société | Émetteur examiné. | Lire la décision. |
| ODD | Objectif de développement durable concerné. | Rattacher la preuve à un objectif. |
| Activité | Activité sur laquelle la lecture s'appuie. | Justifier le lien avec l'objectif. |
| Contribution positive | Élément favorable documenté. | Fonder une entrée. |
| Contribution négative | Élément défavorable documenté. | Fonder un refus. Les deux contributions ne se compensent pas. |
| Preuve | Passage, page et date du document. | Rendre la décision vérifiable. |
| Controverse | Fait qui peut remettre en cause l'entrée. | Compléter la lecture des rapports. |
| Statut | Eligible, Not_Eligible, Indeterminate ou Excluded. | Décider l'entrée. Indeterminate n'est pas une entrée. |
| Motif d'exclusion | Tabac, armes conventionnelles, charbon thermique ou jeux d'argent. | Sortir le titre. |
| Source de l'exclusion | Document nommé pour cette société. | Justifier la sortie. |
| Indicateur fondamental | Information comptable publiée, avec sa période, sa valeur et le dépôt dont elle est tirée. | Documenter l'émetteur. |

Le registre d'exclusion retient ces motifs. La méthodologie d'éligibilité ne fixe pas de seuil de chiffre d'affaires. Aucun seuil n'est ajouté par l'architecture.


### 4.6 Comparaison et journal

| Champ | Description | Utilisation |
|---|---|---|
| Mois | Mois de la comparaison. | Aligner le portefeuille et le fonds. |
| Rendement publié | Rendement du fonds publié par iShares. | Constituer le point de comparaison. |
| Rendement en francs | Ce rendement converti au cours de la Banque nationale suisse. | Comparer le portefeuille et le fonds dans la devise du mandat. |
| Date d'exécution | Moment de l'acquisition ou du contrôle. | Reconstituer l'ordre des traitements. |
| Traitement | Acquisition ou préparation. | Savoir quelle étape a produit le fichier. |
| Fichier produit | Nom et emplacement du résultat. | Retrouver le fichier. |
| Statut du traitement | Issue de l'exécution. | Distinguer une exécution aboutie d'une exécution incomplète. |
| Nombre d'enregistrements | Lignes produites. | Contrôler que le fichier a bien été écrit. |


## Dictionnaire

Ce dictionnaire reprend les termes employés dans cette note qui n'y sont pas déjà expliqués. Il n'en ajoute pas.

Les termes de cette note sont définis à l'endroit où ils apparaissent. Le dictionnaire de données, à la section 4, décrit les champs.
